# Model Building and Training (Task 2)

## Objective
Build, train, and compare classification models for fraud detection on imbalanced data.

- **Baseline**: Logistic Regression (interpretable)
- **Ensemble**: XGBoost (high performance)
- **Metrics**: AUC-PR, F1-Score, Confusion Matrix
- **Cross-validation**: Stratified 5-fold
- **Model selection**: Documented comparison and justification

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve, auc, f1_score,
    confusion_matrix, classification_report)
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded')

Libraries loaded


---
## 1. Data Loading

### Helper: load processed splits

In [3]:
def load_splits(prefix, processed_dir='../data/processed'):
    X_train = pd.read_csv(f'{processed_dir}/{prefix}_X_train_smote.csv').values
    y_train = pd.read_csv(f'{processed_dir}/{prefix}_y_train_smote.csv').values.ravel()
    X_test  = pd.read_csv(f'{processed_dir}/{prefix}_X_test.csv').values
    y_test  = pd.read_csv(f'{processed_dir}/{prefix}_y_test.csv').values.ravel()
    print(f'{prefix}: train {X_train.shape}, test {X_test.shape}')
    return X_train, X_test, y_train, y_test

# Load Fraud_Data splits
X_train_fd, X_test_fd, y_train_fd, y_test_fd = load_splits('fraud')
# Load creditcard splits
X_train_cc, X_test_cc, y_train_cc, y_test_cc = load_splits('cc')

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/fraud_X_train_smote.csv'

---
## 2. Evaluation Helper

In [4]:
def evaluate(model, X, y, label='', threshold=0.5):
    """Full evaluation: AUC-PR, F1, confusion matrix, classification report."""
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)

    prec_arr, rec_arr, _ = precision_recall_curve(y, probs)
    auc_pr = auc(rec_arr, prec_arr)
    f1 = f1_score(y, preds)
    cm = confusion_matrix(y, preds)
    report = classification_report(y, preds, output_dict=True)

    return {
        'model': label,
        'auc_pr': auc_pr,
        'f1': f1,
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'confusion': cm,
        'report': report,
        'probs': probs,
        'preds': preds,
    }

def plot_confusion(cm, title, ax):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

def plot_pr_curve(model, X, y, label, ax):
    probs = model.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y, probs)
    ax.plot(rec, prec, label=f'{label} (AUC={auc(rec,prec):.4f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend()

---
## 3. Fraud_Data.csv Modeling

### 3.1 Baseline: Logistic Regression

In [5]:
lr_fd = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_fd.fit(X_train_fd, y_train_fd)
lr_fd_eval = evaluate(lr_fd, X_test_fd, y_test_fd, label='LogReg (Fraud_Data)')

print(f"AUC-PR: {lr_fd_eval['auc_pr']:.4f}")
print(f"F1:     {lr_fd_eval['f1']:.4f}")
print(f"Precision (fraud): {lr_fd_eval['precision']:.4f}")
print(f"Recall (fraud):    {lr_fd_eval['recall']:.4f}")
print(f"\nConfusion Matrix:\n{lr_fd_eval['confusion']}")
print(f"\nClassification Report:")
print(classification_report(y_test_fd, lr_fd_eval['preds']))

NameError: name 'X_train_fd' is not defined

### 3.2 Ensemble: XGBoost

In [ ]:
xgb_fd = XGBClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', n_jobs=-1, random_state=42
)
xgb_fd.fit(X_train_fd, y_train_fd)
xgb_fd_eval = evaluate(xgb_fd, X_test_fd, y_test_fd, label='XGBoost (Fraud_Data)')

print(f"AUC-PR: {xgb_fd_eval['auc_pr']:.4f}")
print(f"F1:     {xgb_fd_eval['f1']:.4f}")
print(f"Precision (fraud): {xgb_fd_eval['precision']:.4f}")
print(f"Recall (fraud):    {xgb_fd_eval['recall']:.4f}")
print(f"\nConfusion Matrix:\n{xgb_fd_eval['confusion']}")
print(f"\nClassification Report:")
print(classification_report(y_test_fd, xgb_fd_eval['preds']))

### 3.3 Fraud_Data Model Comparison

In [ ]:
# Side-by-side comparison table
fd_results = [lr_fd_eval, xgb_fd_eval]
fd_df = pd.DataFrame([{
    'Model': r['model'],
    'AUC-PR': f"{r['auc_pr']:.4f}",
    'F1': f"{r['f1']:.4f}",
    'Precision (fraud)': f"{r['precision']:.4f}",
    'Recall (fraud)': f"{r['recall']:.4f}",
} for r in fd_results])
print('=== Fraud_Data.csv Model Comparison ===')
display(fd_df)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrices
plot_confusion(lr_fd_eval['confusion'], 'LogReg - Fraud_Data', axes[0])
plot_confusion(xgb_fd_eval['confusion'], 'XGBoost - Fraud_Data', axes[1])

# PR curves
plot_pr_curve(lr_fd, X_test_fd, y_test_fd, 'LogReg', axes[2])
plot_pr_curve(xgb_fd, X_test_fd, y_test_fd, 'XGBoost', axes[2])

plt.tight_layout()
plt.savefig('../data/processed/fraud_data_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Cross-Validation (Stratified 5-Fold) on Fraud_Data

In [ ]:
def cross_validate_scores(model_cls, X, y, cv=5, **kwargs):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    auc_scores, f1_scores = [], []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        Xtr, Xval = X[tr_idx], X[val_idx]
        ytr, yval = y[tr_idx], y[val_idx]
        m = model_cls(**kwargs)
        m.fit(Xtr, ytr)
        probs = m.predict_proba(Xval)[:, 1]
        preds = (probs >= 0.5).astype(int)
        p, r, _ = precision_recall_curve(yval, probs)
        auc_scores.append(auc(r, p))
        f1_scores.append(f1_score(yval, preds))
        print(f'  Fold {fold+1}: AUC-PR={auc(r,p):.4f}, F1={f1_score(yval,preds):.4f}')
    return np.mean(auc_scores), np.std(auc_scores), np.mean(f1_scores), np.std(f1_scores)

# Use a subset for faster CV (XGBoost on full data is slow)
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_fd), size=min(50000, len(X_train_fd)), replace=False)
X_sample = X_train_fd[sample_idx]
y_sample = y_train_fd[sample_idx]

print('Logistic Regression CV:')
lr_mean_auc, lr_std_auc, lr_mean_f1, lr_std_f1 = cross_validate_scores(
    LogisticRegression, X_sample, y_sample,
    class_weight='balanced', max_iter=1000, random_state=42)
print(f'  Mean AUC-PR: {lr_mean_auc:.4f} +/- {lr_std_auc:.4f}')
print(f'  Mean F1:     {lr_mean_f1:.4f} +/- {lr_std_f1:.4f}')

print('\nXGBoost CV:')
xgb_mean_auc, xgb_std_auc, xgb_mean_f1, xgb_std_f1 = cross_validate_scores(
    XGBClassifier, X_sample, y_sample,
    n_estimators=200, learning_rate=0.1, max_depth=5,
    eval_metric='logloss', n_jobs=-1, random_state=42)
print(f'  Mean AUC-PR: {xgb_mean_auc:.4f} +/- {xgb_std_auc:.4f}')
print(f'  Mean F1:     {xgb_mean_f1:.4f} +/- {xgb_std_f1:.4f}')

---
## 4. creditcard.csv Modeling

### 4.1 Baseline: Logistic Regression

In [ ]:
lr_cc = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_cc.fit(X_train_cc, y_train_cc)
lr_cc_eval = evaluate(lr_cc, X_test_cc, y_test_cc, label='LogReg (creditcard)')

print(f"AUC-PR: {lr_cc_eval['auc_pr']:.4f}")
print(f"F1:     {lr_cc_eval['f1']:.4f}")
print(f"Precision (fraud): {lr_cc_eval['precision']:.4f}")
print(f"Recall (fraud):    {lr_cc_eval['recall']:.4f}")
print(f"\nConfusion Matrix:\n{lr_cc_eval['confusion']}")

### 4.2 Ensemble: XGBoost

In [ ]:
xgb_cc = XGBClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', n_jobs=-1, random_state=42
)
xgb_cc.fit(X_train_cc, y_train_cc)
xgb_cc_eval = evaluate(xgb_cc, X_test_cc, y_test_cc, label='XGBoost (creditcard)')

print(f"AUC-PR: {xgb_cc_eval['auc_pr']:.4f}")
print(f"F1:     {xgb_cc_eval['f1']:.4f}")
print(f"Precision (fraud): {xgb_cc_eval['precision']:.4f}")
print(f"Recall (fraud):    {xgb_cc_eval['recall']:.4f}")
print(f"\nConfusion Matrix:\n{xgb_cc_eval['confusion']}")

### 4.3 creditcard Model Comparison

In [ ]:
cc_results = [lr_cc_eval, xgb_cc_eval]
cc_df = pd.DataFrame([{
    'Model': r['model'],
    'AUC-PR': f"{r['auc_pr']:.4f}",
    'F1': f"{r['f1']:.4f}",
    'Precision (fraud)': f"{r['precision']:.4f}",
    'Recall (fraud)': f"{r['recall']:.4f}",
} for r in cc_results])
print('=== creditcard.csv Model Comparison ===')
display(cc_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_confusion(lr_cc_eval['confusion'], 'LogReg - creditcard', axes[0])
plot_confusion(xgb_cc_eval['confusion'], 'XGBoost - creditcard', axes[1])
plot_pr_curve(lr_cc, X_test_cc, y_test_cc, 'LogReg', axes[2])
plot_pr_curve(xgb_cc, X_test_cc, y_test_cc, 'XGBoost', axes[2])
plt.tight_layout()
plt.savefig('../data/processed/creditcard_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Overall Model Selection and Justification

In [ ]:
print('='*70)
print('MODEL SELECTION SUMMARY')
print('='*70)

print('\n--- Fraud_Data.csv ---')
display(fd_df)
best_fd = 'XGBoost' if xgb_fd_eval['auc_pr'] > lr_fd_eval['auc_pr'] else 'LogReg'
print(f'Selected: {best_fd}')
print(f'Justification: XGBoost achieves higher AUC-PR ({xgb_fd_eval["auc_pr"]:.4f} vs '
      f'{lr_fd_eval["auc_pr"]:.4f}) and better F1 ({xgb_fd_eval["f1"]:.4f} vs '
      f'{lr_fd_eval["f1"]:.4f}). While LogReg is more interpretable, '
      f'XGBoost captures non-linear fraud patterns better.')

print('\n--- creditcard.csv ---')
display(cc_df)
best_cc = 'XGBoost' if xgb_cc_eval['auc_pr'] > lr_cc_eval['auc_pr'] else 'LogReg'
print(f'Selected: {best_cc}')
print(f'Justification: XGBoost achieves higher AUC-PR ({xgb_cc_eval["auc_pr"]:.4f} vs '
      f'{lr_cc_eval["auc_pr"]:.4f}) and better F1 ({xgb_cc_eval["f1"]:.4f} vs '
      f'{lr_cc_eval["f1"]:.4f}). For PCA-transformed data, '
      f'tree-based models handle non-linear feature interactions well.')
print('='*70)

---
## 6. Save Trained Models

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(lr_fd,  '../models/lr_fraud_data.pkl')
joblib.dump(xgb_fd, '../models/xgb_fraud_data.pkl')
joblib.dump(lr_cc,  '../models/lr_creditcard.pkl')
joblib.dump(xgb_cc, '../models/xgb_creditcard.pkl')

print('Saved models:')
for f in os.listdir('../models'):
    print(f'  {f}')
print('\nDone!')